# A2 — Retrieval-Augmented Generation (RAG)

**System:** Beacon — Social Media Brand Monitoring  
**Brand:** OpenAI  
**Owner:** Chieri Ishikawa  

---

## 1. Objective

RAG (Retrieval-Augmented Generation) grounds LLM output in real, retrieved evidence rather than model memory alone. In Beacon, A2 serves as the **evidence layer**: when A5 detects a crisis signal or an analyst submits a query, A2 retrieves the most semantically relevant Reddit posts from the OpenAI corpus. These posts are then passed to A1's `run_llm()` for grounded report generation.

Without RAG: the LLM answers from parametric memory — risk of hallucination (Lewis et al., 2021).  
With RAG: the LLM reads real retrieved posts first — factual, evidenced output.

**This notebook addresses three experimental questions:**
1. Which retrieval method produces better results: keyword-based (TF-IDF) or semantic (FAISS + embedding)?
2. Which embedding model produces better retrieval quality: `all-MiniLM-L6-v2` (local) vs `text-embedding-3-small` (OpenAI API)?
3. Does corpus unit choice matter: `text_with_comments` vs `text_raw` only?

**Scope of this notebook (A2 only):**
- Build and persist the FAISS vector index
- Implement `rag_retrieve(query, top_k)` — the exported contract
- Baseline vs semantic retrieval comparison (Recall@5)
- Embedder comparison on same 20 test queries
- Stub LLM demo (generation is A1's responsibility in production)

**Reference:** Lewis, P., Perez, E., Piktus, A., et al. (2021). Retrieval-augmented generation for knowledge-intensive NLP tasks. *arXiv*. https://doi.org/10.48550/arXiv.2005.11401

---
## 2. Setup

In [1]:
# Install dependencies
# !pip install sentence-transformers faiss-cpu numpy pandas scikit-learn --quiet

In [2]:
import sys
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

_root = Path.cwd().parent 
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

DATA_DIR  = _root / "data"
INDEX_DIR = DATA_DIR / "faiss_index"

from shared.data_loader import load_sample
from shared.preprocessing import clean_for_llm
from shared.rag import rag_retrieve as shared_rag_retrieve

# load all Reddit posts for OpenAI
df = load_sample(brand="openai")
print(f"\nDataFrame shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

  reddit_openai_20260515.jsonl: +1030 posts
  reddit_openai_20260516.jsonl: +181 posts
Total: 1211 unique posts across 2 snapshot(s)

DataFrame shape: (1211, 11)
Columns: ['post_id', 'title', 'selftext', 'text', 'subreddit', 'score', 'num_comments', 'created_utc', 'url', 'image_url', 'top_comments']


,post_id,title,selftext,text,subreddit,score,num_comments,created_utc,url,image_url,top_comments
0,1i75wo0,We doing this as well?,,We doing this as well?,OpenAI,28970,1193,2025-01-22 07:35:58+00:00,https://reddit.com/r/OpenAI/comments/1i75wo0/w...,https://i.redd.it/pup9mous0iee1.png,"[{'comment_id': 'm8i4ha3', 'body': 'We could a..."
1,1rgrccs,The end of GPT,,The end of GPT,OpenAI,24819,2861,2026-02-28 03:02:59+00:00,https://reddit.com/r/OpenAI/comments/1rgrccs/t...,https://i.redd.it/0l4hxonfi5mg1.png,"[{'comment_id': 'o7tiuuk', 'body': '""Should I ..."
2,1imhi9l,Offer declined,,Offer declined,OpenAI,15750,550,2025-02-10 21:23:19+00:00,https://reddit.com/r/OpenAI/comments/1imhi9l/o...,https://i.redd.it/opisyw5ppdie1.png,"[{'comment_id': 'mc368q6', 'body': 'Bully is l..."


---
## 3. Implementation

The implementation is split into two phases that mirror the RAG architecture:

- **Phase A (offline, run once):** prepare corpus → embed → build FAISS index → save to disk
- **Phase B (online, called at query time):** load index → embed query → search → return top-k posts

### 3A — Corpus Preparation

In [3]:
# 3A-1: PREPARE CORPUS
def build_text_with_comments(row) -> str:
    parts = []

    if pd.notna(row.get("title")) and str(row["title"]).strip():
        parts.append(str(row["title"]).strip())

    if pd.notna(row.get("selftext")) and str(row["selftext"]).strip() not in ("", "[removed]", "[deleted]"):
        parts.append(str(row["selftext"]).strip())

    comments = row.get("top_comments", [])

    # Handle case where JSONL loaded top_comments as a string
    if isinstance(comments, str):
        try:
            comments = json.loads(comments)
        except Exception:
            comments = []

    # Data snapshot structure: list of {"comment_id": ..., "body": ..., "score": ...}
    if isinstance(comments, list):
        for c in comments[:5]:
            body = c.get("body", "") if isinstance(c, dict) else str(c)
            if body.strip() not in ("", "[removed]", "[deleted]"):
                parts.append(body.strip())

    return " | ".join(parts)

# Build the corpus by combining title, selftext, and top comments
corpus_df = df.copy().reset_index(drop=True)
corpus_df["text_with_comments"] = corpus_df.apply(build_text_with_comments, axis=1)

# Drop rows where the combined text is still empty
corpus_df = corpus_df[corpus_df["text_with_comments"].str.strip() != ""].reset_index(drop=True)
raw_texts = corpus_df["text_with_comments"].tolist()

print(f"Corpus size: {len(raw_texts)} posts")
print(f"\nExample (first post):")
print(raw_texts[0][:400])

Corpus size: 1211 posts

Example (first post):
We doing this as well? | We could allow Screenshots but bann links. So they dont gather traffic on their posts. And once one shot was posted, duplicates will be deleted | I believe in an open and free society where all people are equal. Nothing less, except nazis. Fuck nazis. Fuck nazis. Ban all x / twitter. | Sure, censor things. That's always a good path to take. | No, this is a sub about OpenAI


In [4]:
# 3A-2: Apply clean_for_llm
# Strips URLs, emojis, Reddit markdown, usernames using preprocessing function.
print("Cleaning corpus...")
t0 = time.time()
clean_texts = clean_for_llm(raw_texts)
print(f"Done in {time.time()-t0:.1f}s")

print(f"\nCleaned (first post, first 300 chars):")
print(clean_texts[0][:300])

Cleaning corpus...
Done in 0.8s

Cleaned (first post, first 300 chars):
We doing this as well? | We could allow Screenshots but bann links. So they do not gather traffic on their posts. And once one shot was posted, duplicates will be deleted | I believe in an open and free society where all people are equal. Nothing less, except nazis. Fuck nazis. Fuck nazis. Ban all x


In [5]:
# Spot check: confirm URLs and emojis are gone
import re

url_pattern   = re.compile(r"https?://")
emoji_pattern = re.compile(u"[\U00010000-\U0010ffff]", flags=re.UNICODE)

has_url   = sum(1 for t in clean_texts if url_pattern.search(t))
has_emoji = sum(1 for t in clean_texts if emoji_pattern.search(t))

print(f"Posts still containing URLs:  {has_url} / {len(clean_texts)}")
print(f"Posts still containing emoji: {has_emoji} / {len(clean_texts)}")
print(f"Avg cleaned text length: {sum(len(t) for t in clean_texts) / len(clean_texts):.0f} chars")

Posts still containing URLs:  0 / 1211
Posts still containing emoji: 3 / 1211
Avg cleaned text length: 1379 chars


### 3B — Embedding and Index Build (Primary Method: FAISS + all-MiniLM-L6-v2)

In [6]:
# 3B-1: Embed corpus with all-MiniLM-L6-v2
from sentence_transformers import SentenceTransformer
import numpy as np
import time

model = SentenceTransformer("all-MiniLM-L6-v2")

t0 = time.time()
corpus_embeddings = model.encode(
    clean_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)
embed_time_minilm = time.time() - t0

print(f"Shape: {corpus_embeddings.shape}")
print(f"Time: {embed_time_minilm:.1f}s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Shape: (1211, 384)
Time: 7.9s


In [7]:
# 3B-2: Embed corpus with multi-qa-MiniLM-L6-cos-v1
model_qa = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")

t0 = time.time()
corpus_embeddings_qa = model_qa.encode(
    clean_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)
embed_time_qa = time.time() - t0

print(f"Shape: {corpus_embeddings_qa.shape}")
print(f"Time: {embed_time_qa:.1f}s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Shape: (1211, 384)
Time: 10.3s


In [8]:
# 3B-3: Build FAISS index and evaluate retrieval on test queries
import faiss
import time
import pandas as pd

test_queries = [
    "OpenAI pricing too expensive",
    "ChatGPT outage service down",
    "GPT-4 hallucination wrong answers",
    "OpenAI safety alignment concerns",
    "Sam Altman leadership controversy",
]

def build_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index

def evaluate_model(model, embeddings, texts, queries, top_k=5):
    index = build_index(embeddings)
    rows = []

    for q in queries:
        cleaned_q = clean_for_llm([q])[0]

        t0 = time.time()
        q_vec = model.encode([cleaned_q], normalize_embeddings=True)
        scores, indices = index.search(q_vec, top_k)
        latency_ms = (time.time() - t0) * 1000

        top_scores = scores[0]
        top_texts  = [texts[i] for i in indices[0] if i != -1]

        rows.append({
            "query":          q,
            "top1_score":     round(float(top_scores[0]), 4),
            "mean_top5_score": round(float(top_scores.mean()), 4),
            "latency_ms":     round(latency_ms, 2),
            "top1_result":    top_texts[0][:150] if top_texts else "",
        })

    return pd.DataFrame(rows)


results_minilm = evaluate_model(model,    corpus_embeddings,    clean_texts, test_queries)
results_qa     = evaluate_model(model_qa, corpus_embeddings_qa, clean_texts, test_queries)

print("=== all-MiniLM-L6-v2 ===")
print(results_minilm.to_string(index=False))

print("\n=== multi-qa-MiniLM-L6-cos-v1 ===")
print(results_qa.to_string(index=False))

=== all-MiniLM-L6-v2 ===
                            query  top1_score  mean_top5_score  latency_ms                                                                                                                                            top1_result
     OpenAI pricing too expensive      0.6650           0.6068     1290.13 OpenAI is losing money | Wow, and here I thought $200 would be the break even price. | I am one of the pro sub. I use a lot. | They have always lost m
      ChatGPT outage service down      0.6507           0.5244      262.97 Is chat GPT down or is it just me? | I get this no matter what I send. I have tried everything and nothing seems to fix it. | Someone just asked me ho
GPT-4 hallucination wrong answers      0.5646           0.4982      285.10 I had no idea GPT could realise it was wrong | I had a similar situation when asking it to write some code. The answer it produced was mostly right, b
 OpenAI safety alignment concerns      0.5314           0.5087       11

In [9]:
# Aggregate comparison
summary = pd.DataFrame([
    {
        "model":              "all-MiniLM-L6-v2",
        "mean_top1_score":    round(results_minilm["top1_score"].mean(), 4),
        "mean_top5_score":    round(results_minilm["mean_top5_score"].mean(), 4),
        "mean_latency_ms":    round(results_minilm["latency_ms"].mean(), 2),
        "embed_time_s":       round(embed_time_minilm, 2),
    },
    {
        "model":              "multi-qa-MiniLM-L6-cos-v1",
        "mean_top1_score":    round(results_qa["top1_score"].mean(), 4),
        "mean_top5_score":    round(results_qa["mean_top5_score"].mean(), 4),
        "mean_latency_ms":    round(results_qa["latency_ms"].mean(), 2),
        "embed_time_s":       round(embed_time_qa, 2),
    },
])

print("=== Aggregate Comparison (computed) ===")
print(summary.to_string(index=False))

=== Aggregate Comparison (computed) ===
                    model  mean_top1_score  mean_top5_score  mean_latency_ms  embed_time_s
         all-MiniLM-L6-v2           0.6000           0.5377           372.03          7.89
multi-qa-MiniLM-L6-cos-v1           0.5999           0.5546             7.95         10.30


Based on the evaluation results, multi-qa-MiniLM-L6-cos-v1 was selected as the primary embedding model for the RAG pipeline. Unlike all-MiniLM-L6-v2, which was trained on general sentence similarity tasks, multi-qa-MiniLM-L6-cos-v1 was specifically optimised for question-answer retrieval, which directly mirrors the RAG use case where analyst queries and crisis signals function as questions posed against a corpus of Reddit posts. This alignment between training objective and deployment task produced a higher mean top-5 relevance score (0.5546 vs 0.5377) and a query latency of 8.79ms — approximately six times faster than the general-purpose alternative — making it better suited for a real-time brand monitoring system where retrieval is triggered on demand by crisis detection signals and must return grounded evidence within interactive response times.

In [10]:
# 3B-4: Persist FAISS index and corpus texts to disk
import faiss
import json

INDEX_DIR = DATA_DIR / "faiss_index"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

# build index from the winning model's embeddings
dim   = corpus_embeddings_qa.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(corpus_embeddings_qa)

print(f"Index type:     IndexFlatIP")
print(f"Dimension:      {dim}")
print(f"Vectors stored: {index.ntotal}")

# save index to disk
faiss.write_index(index, str(INDEX_DIR / "corpus.index"))

# save corpus texts — parallel list: position i → post text
with open(INDEX_DIR / "corpus_texts.json", "w", encoding="utf-8") as f:
    json.dump(clean_texts, f, ensure_ascii=False)

# save post IDs for traceability
post_ids = corpus_df["post_id"].tolist()
with open(INDEX_DIR / "corpus_ids.json", "w", encoding="utf-8") as f:
    json.dump(post_ids, f)

print(f"\nSaved to {INDEX_DIR}/")
print(f"  corpus.index       — FAISS binary")
print(f"  corpus_texts.json  — {len(clean_texts)} post texts")
print(f"  corpus_ids.json    — {len(post_ids)} post IDs")

Index type:     IndexFlatIP
Dimension:      384
Vectors stored: 1211

Saved to /Users/chi/workspace/beacon/data/faiss_index/
  corpus.index       — FAISS binary
  corpus_texts.json  — 1211 post texts
  corpus_ids.json    — 1211 post IDs


### 3C — Retrieval Function (Phase B: Online)

In [11]:
# --- Load saved index (this is what runs at demo time) ---
# Building the index is offline. At query time, we load from disk.

index = faiss.read_index(str(INDEX_DIR / "corpus.index"))

with open(INDEX_DIR / "corpus_texts.json", encoding="utf-8") as f:
    corpus_texts = json.load(f)

print(f"Index loaded: {index.ntotal} vectors")
print(f"Corpus texts: {len(corpus_texts)} entries")

Index loaded: 1211 vectors
Corpus texts: 1211 entries


In [12]:
# --- Define the primary rag_retrieve function ---
# This is the exported contract. Signature must not change.

def rag_retrieve(query: str, top_k: int = 5) -> list[str]:
    """
    Retrieve the top-k most semantically relevant Reddit posts for a query.

    Uses FAISS IndexFlatIP with all-MiniLM-L6-v2 embeddings.
    Preprocessing is applied symmetrically to corpus and query.

    Parameters
    ----------
    query  : str  — plain text question or crisis signal
    top_k  : int  — number of posts to return (default 5)

    Returns
    -------
    list[str] — post texts ordered by cosine similarity, most relevant first
    """
    # Step 1: clean the query identically to how the corpus was cleaned
    cleaned  = clean_for_llm([query])[0]

    # Step 2: embed the query with the same model as the corpus
    q_vec    = model_qa.encode([cleaned], normalize_embeddings=True)


    # Step 3: FAISS search — returns (scores, indices) arrays of shape (1, top_k)
    scores, indices = index.search(q_vec, top_k)

    # Step 4: map indices back to post texts
    results  = [corpus_texts[i] for i in indices[0] if i != -1]

    return results, scores[0]

In [13]:
query = "OpenAI pricing too expensive"
results, scores = rag_retrieve(query, top_k=5)

print(f"Query: {query}\n")
for i, (post, score) in enumerate(zip(results, scores), 1):
    print(f"[{i}] Score: {score:.4f}")
    print(f"     {post[:200]}")
    print()

Query: OpenAI pricing too expensive

[1] Score: 0.6658
     OpenAI Could Be Blowing As Much As $15 Million Per Day On Silly Sora Videos | Some back-of-napkin math suggests OpenAI is spending more than a quarter of what it is making to power its AI slop factory

[2] Score: 0.6489
     OpenAI's inspiration. | You using last year's numbers? Revenue for OpenAI is looking to be $13B this year | Oh man, billions here and there. They are even talking about trillions now. and then I open 

[3] Score: 0.6482
     OpenAI is losing money | Wow, and here I thought $200 would be the break even price. | I am one of the pro sub. I use a lot. | They have always lost money. Granted it is other people’s money but they 

[4] Score: 0.6198
     OpenAI When the Free Market Hits Back | tie spark aback elderly boat hospital dam shocking retire recognise This post was mass deleted and anonymized with Redact | It is not even just china, hell, Gem

[5] Score: 0.5886
     OpenAI profit | I saw this on LinkedIn, 

### 3D - Trying rag_retrieve() with a Ollama qwen2.5:3b

In [14]:
import subprocess
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(result.stdout)

NAME          ID              SIZE      MODIFIED    
qwen2.5:3b    357c53fb659c    1.9 GB    6 weeks ago    



In [15]:
# Generate with rag using qwen2.5:3b
import requests

def generate_with_rag(query, top_k=5):
    results, scores = rag_retrieve(query, top_k=top_k)
    
    context = "\n\n".join(
        [f"[Post {i+1}]: {post[:200]}" for i, post in enumerate(results)]
    )
    
    prompt = f"""You are a brand analyst. Using only the Reddit posts below, answer the question briefly.

                Reddit Posts:
                {context}

                Question: {query}
                Answer:"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model":  "qwen2.5:3b",
            "prompt": prompt,
            "stream": False
        },
        timeout=120
    )
    
    return response.json()["response"].strip()


# debug: confirm Ollama API is reachable first
ping = requests.get("http://localhost:11434")
print(f"Ollama status: {ping.status_code}")  # should print 200

query  = "Why are users concerned about OpenAI pricing?"
answer = generate_with_rag(query)
print(f"Query: {query}\n")
print(f"Answer:\n{answer}")

Ollama status: 200
Query: Why are users concerned about OpenAI pricing?

Answer:
Users seem to be concerned about OpenAI's pricing strategy due to several reasons mentioned in the Reddit posts:

1. **High Revenue and Potential for Trillions**: The first post suggests that OpenAI is looking at a revenue of $13 billion this year, which has led some users to speculate even higher numbers (possibly trillions). This high level of potential revenue seems alarming to users who might be concerned about any pricing strategy that could result in such significant profits.

2. **Limited Access Despite Availability**: The second post indicates that despite the models being available publicly, they are accessible through a paid API. Users are questioning why this is so, implying concerns over whether the service should be free or if it's just not widely known or utilized.

3. **High Spending on Unrelated Activities**: The third post reveals that OpenAI might be spending more than $15 million per day

In [16]:
# TODO: Check the speed, and if model is hallucinating by comparing answer against retrieved posts. Look for unsupported claims or details not present in the posts.
# TODO: try to create agentic rag


---
## 4. Results

This section shows example retrieval outputs and the Recall@5 evaluation on 20 test queries.

### 4A — Example Retrieval Outputs

In [17]:
# --- Run 5 representative queries and show top result for each ---
demo_queries = [
    "OpenAI GPT-4o pricing too expensive",
    "ChatGPT outage users cannot access",
    "OpenAI safety concerns alignment risk",
    "Sam Altman fired board drama",
    "GPT-4 hallucination wrong answers",
]

for q in demo_queries:
    results = rag_retrieve(q, top_k=5)
    print(f"QUERY: {q}")
    print(f"TOP RESULT: {results[0][:250]}")
    print("-" * 80)

QUERY: OpenAI GPT-4o pricing too expensive
TOP RESULT: ['Sam Altman admits OpenAI ‘totally screwed up’ its GPT-5 launch and says the company will spend trillions of dollars on data centers | So basically the $20~billion in funding they get for next years to ‘keep the lights on’ with the current model services will fall way short of the trillion they need to ramp up capacity to reach demand? Either GPT subscriptions need to get much more expensive or they will have to pull the plug on free access to force AI junkies to start paying (full disclosure I pay for for GPT plus and have 2x Github copilot subs) But that only works if EVERYONE cuts free access. | I am desperately trying to get literally any job at OAI so I can rapidly takeover their sales & marketing leadership personally. It is absolutely embarrassing. Their corporate release videos feel alien, uncanny valley. It is so bad and so solvable. Such a bizarre time to watch an incredible, life changing product get communicated and so

### 4B — Recall@5 Evaluation

Recall@5 answers: for a given query, does the expected relevant post appear in the top 5 results?

**How to construct test pairs:** for each query below, manually identify a post from the corpus that is clearly relevant (use the post_id from `corpus_ids.json`). You are building a small ground-truth set of 20 pairs.

In [18]:
# --- Load post IDs for ground truth matching ---
with open(INDEX_DIR / "corpus_ids.json") as f:
    corpus_ids = json.load(f)

def recall_at_k(test_pairs: list[dict], retrieval_fn, k: int = 5) -> float:
    """
    Compute Recall@K over a set of query-ground_truth_post_id pairs.

    Parameters
    ----------
    test_pairs    : list of {"query": str, "expected_post_id": str}
    retrieval_fn  : function matching rag_retrieve signature
    k             : number of results to check

    Returns
    -------
    float — proportion of queries where the expected post appeared in top-k
    """
    hits = 0
    for pair in test_pairs:
        retrieved_texts = retrieval_fn(pair["query"], top_k=k)
        # Map retrieved texts back to their post IDs
        retrieved_ids = set()
        for text in retrieved_texts:
            if text in corpus_texts_loaded:
                idx = corpus_texts_loaded.index(text)
                if idx < len(corpus_ids):
                    retrieved_ids.add(str(corpus_ids[idx]))
        if str(pair["expected_post_id"]) in retrieved_ids:
            hits += 1
    return hits / len(test_pairs) if test_pairs else 0.0


# --- FILL IN: 20 manually selected query-post_id pairs ---
# To find post IDs: after loading corpus_df, inspect df[["post_id", "text_with_comments"]]
# and select posts that clearly match each query topic.
# Replace "FILL_POST_ID_HERE" with actual IDs from your corpus_ids.json

TEST_PAIRS = [
    {"query": "OpenAI GPT-4o pricing too expensive",        "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "ChatGPT outage service unavailable",         "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "OpenAI safety alignment AI risk",            "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "Sam Altman board fired drama",               "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "GPT-4 hallucination incorrect answers",      "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "ChatGPT subscription worth the cost",        "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "OpenAI data privacy concerns users",         "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "GPT-4o vision multimodal capability",        "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "OpenAI competitor Anthropic Claude",         "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "ChatGPT job replacement fear workforce",     "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "OpenAI API rate limit developer complaints", "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "GPT-4 worse performance degraded",           "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "OpenAI jailbreak prompt injection",          "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "ChatGPT memory personalisation feature",     "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "OpenAI Sora video generation model",         "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "AI replacing programmers software jobs",     "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "ChatGPT plagiarism academic integrity",      "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "OpenAI Microsoft partnership Copilot",       "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "GPT-4 creative writing capability fiction",  "expected_post_id": "FILL_POST_ID_HERE"},
    {"query": "OpenAI AGI artificial general intelligence",  "expected_post_id": "FILL_POST_ID_HERE"},
]

print(f"Test set: {len(TEST_PAIRS)} query-post pairs")
print("Note: replace 'FILL_POST_ID_HERE' with actual post IDs from your corpus before running evaluation.")

Test set: 20 query-post pairs
Note: replace 'FILL_POST_ID_HERE' with actual post IDs from your corpus before running evaluation.


---
## 5. Comparison — Baseline vs Semantic vs Embedder

Three experiments are run on the same 20 test queries:

| Experiment | Method | Purpose |
|---|---|---|
| E1 | TF-IDF keyword retrieval | **Baseline** — no embedding, exact word matching |
| E2 | FAISS + `all-MiniLM-L6-v2` | **Primary method** — local, no API key |
| E3 | FAISS + `text-embedding-3-small` | **Dev comparison** — OpenAI API, requires key |

### E1 — Baseline: TF-IDF Retrieval

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- Build TF-IDF matrix over the same cleaned corpus ---
# TF-IDF represents each post as a weighted bag-of-words vector.
# Retrieval is cosine similarity between query vector and corpus vectors.
# This is the standard IR baseline before dense/semantic methods.

print("Building TF-IDF matrix...")
t0 = time.time()
tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(clean_texts)   # shape: (n_posts, vocab_size)
print(f"Done in {time.time()-t0:.1f}s")
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")


def tfidf_retrieve(query: str, top_k: int = 5) -> list[str]:
    """Baseline retrieval: TF-IDF keyword matching."""
    cleaned_query = clean_for_llm([query])[0]
    query_vec = tfidf.transform([cleaned_query])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = scores.argsort()[::-1][:top_k]
    return [clean_texts[i] for i in top_indices]


# Quick test
baseline_results = tfidf_retrieve("OpenAI GPT-4o pricing too expensive", top_k=3)
print(f"\nBaseline top result (first 300 chars):")
print(baseline_results[0][:300])

Building TF-IDF matrix...
Done in 0.4s
TF-IDF matrix shape: (1211, 15000)

Baseline top result (first 300 chars):
Mr president the second chinese ai has hit the market | It is the new Qwen 2.5 Max model, which has no "thinking mode", is not open source and super expensive to use in the API. 3-4x more expensive than GPT 4o: Qwen 2.5 Max: $10/M input tokens, $30/M output tokens GPT-4o: $2.50/M input and $10/M out


### E2 — Primary Method: FAISS + all-MiniLM-L6-v2

Already implemented above as `rag_retrieve()`. This cell measures latency for the comparison table.

In [20]:
# --- Measure latency for all-MiniLM-L6-v2 ---
queries_for_timing = [p["query"] for p in TEST_PAIRS[:10]]

t0 = time.time()
for q in queries_for_timing:
    rag_retrieve(q, top_k=5)
latency_minilm = (time.time() - t0) / len(queries_for_timing) * 1000   # ms per query

t0 = time.time()
for q in queries_for_timing:
    tfidf_retrieve(q, top_k=5)
latency_tfidf = (time.time() - t0) / len(queries_for_timing) * 1000   # ms per query

print(f"Avg latency per query:")
print(f"  TF-IDF baseline:       {latency_tfidf:.1f} ms")
print(f"  FAISS + MiniLM:        {latency_minilm:.1f} ms")

Avg latency per query:
  TF-IDF baseline:       1.9 ms
  FAISS + MiniLM:        11.6 ms


### E3 — Embedder Comparison: text-embedding-3-small (OpenAI API)

This arm requires an OpenAI API key. It is a **development-only comparison** — never used in the production demo path. Skip this cell if `OPENAI_API_KEY` is not set.

In [21]:
# --- E3: OpenAI text-embedding-3-small (dev comparison only) ---
# Requires: pip install openai
# Requires: OPENAI_API_KEY set in shared/config.py or environment

import os
OPENAI_KEY = os.environ.get("OPENAI_API_KEY", "")

if not OPENAI_KEY:
    print("OPENAI_API_KEY not set — skipping E3 embedder comparison.")
    print("Set the key to run this comparison arm.")
    recall_openai = None
    latency_openai = None
else:
    from openai import OpenAI
    openai_client = OpenAI(api_key=OPENAI_KEY)

    def openai_embed(texts: list[str], model="text-embedding-3-small") -> np.ndarray:
        """Embed a list of texts using OpenAI API. Returns L2-normalised array."""
        resp = openai_client.embeddings.create(input=texts, model=model)
        vecs = np.array([d.embedding for d in resp.data], dtype="float32")
        # L2-normalise so IndexFlatIP == cosine
        norms = np.linalg.norm(vecs, axis=1, keepdims=True)
        return vecs / np.where(norms == 0, 1, norms)

    print("Embedding corpus with text-embedding-3-small...")
    print("Warning: embeds in batches to avoid rate limits. May take several minutes.")
    BATCH = 100
    openai_vecs = []
    t0 = time.time()
    for start in range(0, len(clean_texts), BATCH):
        batch = clean_texts[start:start+BATCH]
        openai_vecs.append(openai_embed(batch))
        if (start // BATCH) % 5 == 0:
            print(f"  {start+len(batch)}/{len(clean_texts)} posts embedded")

    corpus_embeddings_openai = np.vstack(openai_vecs)
    print(f"Done in {time.time()-t0:.1f}s. Shape: {corpus_embeddings_openai.shape}")

    # Build FAISS index for OpenAI embeddings
    dim_openai = corpus_embeddings_openai.shape[1]   # 1536 for text-embedding-3-small
    index_openai = faiss.IndexFlatIP(dim_openai)
    index_openai.add(corpus_embeddings_openai)

    def openai_retrieve(query: str, top_k: int = 5) -> list[str]:
        cleaned_query = clean_for_llm([query])[0]
        q_vec = openai_embed([cleaned_query])
        _, indices = index_openai.search(q_vec, top_k)
        return [clean_texts[i] for i in indices[0] if i != -1]

    print("OpenAI retrieval function ready.")

OPENAI_API_KEY not set — skipping E3 embedder comparison.
Set the key to run this comparison arm.


In [22]:
# --- Run Recall@5 for all three methods ---
# NOTE: This cell requires TEST_PAIRS to have actual post IDs filled in.
# Run after completing the ground truth annotation step above.

# Uncomment and run after filling in TEST_PAIRS:
# recall_tfidf  = recall_at_k(TEST_PAIRS, tfidf_retrieve,  k=5)
# recall_minilm = recall_at_k(TEST_PAIRS, rag_retrieve,    k=5)
# recall_openai = recall_at_k(TEST_PAIRS, openai_retrieve, k=5) if OPENAI_KEY else None

# --- Placeholder results table (replace with actual numbers) ---
results_table = pd.DataFrame([
    {
        "Method":        "TF-IDF (Baseline)",
        "Recall@5":      "RUN_AND_FILL",
        "Avg Latency (ms)": f"{latency_tfidf:.1f}",
        "API Key Required": "No",
        "Cost":          "Free",
        "Embedding Dim": "Vocab-size sparse",
    },
    {
        "Method":        "FAISS + all-MiniLM-L6-v2 (Primary)",
        "Recall@5":      "RUN_AND_FILL",
        "Avg Latency (ms)": f"{latency_minilm:.1f}",
        "API Key Required": "No",
        "Cost":          "Free",
        "Embedding Dim": "384 dense",
    },
    {
        "Method":        "FAISS + text-embedding-3-small (Dev)",
        "Recall@5":      "RUN_AND_FILL (skip if no key)",
        "Avg Latency (ms)": "~200 (API round-trip)",
        "API Key Required": "Yes",
        "Cost":          "~$0.02/1M tokens",
        "Embedding Dim": "1536 dense",
    },
])

print("=== Retrieval Comparison Results ===")
results_table

=== Retrieval Comparison Results ===


,Method,Recall@5,Avg Latency (ms),API Key Required,Cost,Embedding Dim
0,TF-IDF (Baseline),RUN_AND_FILL,1.9,No,Free,Vocab-size sparse
1,FAISS + all-MiniLM-L6-v2 (Primary),RUN_AND_FILL,11.6,No,Free,384 dense
2,FAISS + text-embedding-3-small (Dev),RUN_AND_FILL (skip if no key),~200 (API round-trip),Yes,~$0.02/1M tokens,1536 dense


---
## 6. Justification

**Why FAISS + all-MiniLM-L6-v2 over TF-IDF:**
TF-IDF retrieval is a strong baseline for exact keyword matching but fails on semantic similarity. A query such as "pricing too expensive" will not match a post containing "not worth the cost" or "way overpriced" because no tokens are shared. Dense semantic embeddings resolve this by encoding meaning rather than surface form. Lewis et al. (2021) demonstrated that retrieval quality directly bounds generation quality — poor retrieval produces hallucinated outputs even with a high-quality LLM. For a brand monitoring system where crisis signals are phrased differently across users, semantic retrieval is essential.

**Why all-MiniLM-L6-v2 over text-embedding-3-small:**
Both models produce high-quality dense embeddings. `text-embedding-3-small` has a higher-dimensional space (1536 vs 384) and achieves higher scores on the MTEB benchmark. However, it requires an OpenAI API key and incurs per-token cost — both unacceptable for a demo that must run fully offline. `all-MiniLM-L6-v2` runs locally, has zero cost, and completes inference on 5,000 posts in under 3 minutes on CPU. Given the small corpus size, the quality gap between the two models is not material for this use case.

**Why IndexFlatIP over approximate indices (HNSW, IVF):**
Approximate search methods (HNSW, IVF) trade retrieval accuracy for speed. At a corpus size of ~5,000 posts, an exact FAISS search completes in under 10ms per query — the speed gain from approximation is irrelevant. IndexFlatIP is also simpler to reason about, easier to validate, and requires no index training step, which reduces implementation risk.

---
## 7. Limitations

The corpus is limited to two Reddit snapshot dates (May 2026), which constrains temporal coverage and may not reflect current brand perception at demo time. The corpus draws exclusively from three subreddits (r/openai, r/chatgpt, r/artificial), introducing sampling bias toward technically-engaged users. `IndexFlatIP` performs exact linear search and scales as O(n) — for corpora exceeding 100,000 posts, an approximate index (HNSW or IVF) would be required. Retrieval quality is bounded by corpus coverage: if no relevant post exists in the corpus, FAISS will return the least-irrelevant result rather than indicating absence, which can mislead the downstream LLM. Finally, semantic similarity does not guarantee factual relevance — a post that is thematically close to the query but expresses an opposite sentiment will still rank highly.

---
## 8. Pipeline Connection

In the Beacon pipeline, A2 is positioned between the analysis layer (B4, B5, A5) and the generation layer (A1, A3). A5's `detect_crisis()` produces a `crisis_level` and a short crisis description string. That string is passed directly as the `query` argument to `rag_retrieve()`, which returns the five most relevant Reddit posts as a `list[str]`. These posts are then injected into A1's `run_llm()` prompt as grounding context. The full handoff in the LangGraph state schema is: A5 populates `state["crisis_level"]` and `state["crisis_signal"]`; A2 populates `state["retrieved_contexts"]`; A1 reads both and populates `state["insights"]`.

---
## 9. Export Function + Smoke Test

In [23]:
# --- STUB LLM: simulate A1 generation for notebook demo only ---
# In production: replace this with A1's run_llm(prompt) call.
# This stub exists solely to show the full RAG loop inside A2's notebook.

def stub_llm(query: str, contexts: list[str]) -> str:
    """Demo stub — simulates A1 generation. NOT exported. NOT production code."""
    context_str = "\n\n".join(
        [f"[Post {i+1}]: {c[:300]}" for i, c in enumerate(contexts)]
    )
    return (
        f"[STUB — replace with A1 run_llm() in production]\n"
        f"Query: {query}\n\n"
        f"Context passed to LLM:\n{context_str}\n\n"
        f"[LLM would generate a grounded brand intelligence paragraph here]"
    )


# --- Demo: full RAG loop ---
demo_query   = "Why are users upset about OpenAI pricing?"
retrieved    = rag_retrieve(demo_query, top_k=5)
llm_response = stub_llm(demo_query, retrieved)
print(llm_response)

[STUB — replace with A1 run_llm() in production]
Query: Why are users upset about OpenAI pricing?

Context passed to LLM:
[Post 1]: ["OpenAI's inspiration. | You using last year's numbers? Revenue for OpenAI is looking to be $13B this year | Oh man, billions here and there. They are even talking about trillions now. and then I open up my bank account app and see 50 bucks. What is wrong? | OnlyFans is highly profitable, earning around $658 million in pre-tax profit on $1.4 billion revenue in 2024. | They have a goal of replacing most economic creators - and as much as you might not believe it, writing is not the biggest expense, and horny people are not always that picky. | I think they are trying to compete against grok spicy mode.", "OpenAI is losing money | Wow, and here I thought $200 would be the break even price. | I am one of the pro sub. I use a lot. | They have always lost money. Granted it is other people’s money but they keep losing it. | I have pro subscription too and half 

In [ ]:
# Wrapper function for RAG retrieval specific to OpenAI dataset. This is the function that will be called by A1's agentic RAG component. It simply delegates to the shared_rag_retrieve function with the appropriate brand argument.
def rag_retrieve(query: str, top_k: int = 5) -> list[str]:
    return shared_rag_retrieve(query, top_k, brand="openai")